# 03 — Làm sạch dữ liệu quốc gia, nền kinh tế và vùng lãnh thổ

## Mục tiêu và phạm vi dữ liệu

Notebook kết hợp lạm phát CPI hằng năm, GDP theo USD hiện hành và dân số bằng khóa `(iso3, year)`. Bảng phân tích được giới hạn trong giai đoạn **2013–2024**, giữ các mã ISO3 có trong `country_codes.csv`; mã tổng hợp hoặc mã không ánh xạ được được xuất riêng để kiểm tra.

GDP bình quân đầu người được tính từ GDP và dân số của cùng một mã ISO3, cùng năm. Giá trị thiếu không được nội suy. Lạm phát âm được giữ, còn các quan sát lạm phát cực đoan chỉ được gắn cờ để phục vụ mô tả.

Đơn vị quan sát bao gồm quốc gia, nền kinh tế và vùng lãnh thổ theo bảng mã nguồn. Các chỉ tiêu này cung cấp bối cảnh kinh tế; dữ liệu không có thông tin về mức sử dụng tiền điện tử theo quốc gia.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd() / "scripts"))
from common import RAW, TABLES, QUALITY, ensure_output_dirs, assert_unique_key, write_csv_atomic
ensure_output_dirs()

## Tạo bảng thông tin địa lý và giữ đúng mã Bắc Mỹ

`dim_country` lấy mã ISO, tên, khu vực, châu lục, thủ đô, tiền tệ và thông tin nhóm nước kém phát triển từ `country_codes.csv`. Dòng thiếu ISO3 bị loại; nếu ISO3 trùng, chỉ giữ dòng xuất hiện đầu tiên trước khi kiểm tra khóa.

Chuỗi `NA` trong cột châu lục được giữ và ánh xạ thành `North America` — Bắc Mỹ. Việc đọc dữ liệu chỉ coi ô rỗng là thiếu, tránh làm mất nhóm Bắc Mỹ do cách nhận diện giá trị thiếu mặc định.

Sau chuẩn hóa, mỗi ISO3 có một dòng trong `dim_country`, làm cơ sở lọc mã và bổ sung thuộc tính địa lý cho bảng chỉ tiêu theo năm.

In [2]:
cc = pd.read_csv(RAW / "country" / "country_codes.csv", low_memory=False, keep_default_na=False, na_values=[""])
dim_country = cc[[
    "ISO3166-1-Alpha-3","ISO3166-1-Alpha-2","official_name_en","Region Name",
    "Sub-region Name","Continent","Capital","ISO4217-currency_alphabetic_code",
    "Least Developed Countries (LDC)",
]].copy()
dim_country.columns = ["iso3","iso2","country_name","region","sub_region","continent_code","capital","currency_code","ldc_marker"]
dim_country = dim_country.dropna(subset=["iso3"]).drop_duplicates("iso3")
continent_names = {"AF":"Africa","AS":"Asia","EU":"Europe","NA":"North America","SA":"South America","OC":"Oceania","AN":"Antarctica"}
dim_country["continent_name"] = dim_country.continent_code.map(continent_names)
dim_country["is_ldc"] = dim_country.ldc_marker.notna().astype("Int64")
dim_country = dim_country.drop(columns="ldc_marker").sort_values("iso3")
assert_unique_key(dim_country, ["iso3"], "dim_country")
if dim_country.loc[dim_country.continent_code.eq("NA")].empty:
    raise AssertionError("Mất Bắc Mỹ do hiểu sai chuỗi NA là missing")

## Chuẩn hóa chỉ tiêu, nối bảng và lưu kết quả

Cột `CPI` của tệp World Bank được đổi tên thành `inflation_annual_pct` vì chứa tỷ lệ lạm phát hằng năm, không phải mức chỉ số CPI. GDP và dân số được chuyển sang số; giá trị không dương được để thiếu.

Ba nguồn được kiểm tra khóa `(iso3, year)` và nối ngoài với quan hệ một–một. Cách nối này giữ các cặp ISO3–năm xuất hiện trong ít nhất một nguồn, kể cả khi còn thiếu chỉ tiêu khác. Sau khi lọc mã và giai đoạn 2013–2024, thuộc tính địa lý được nối theo ISO3 với quan hệ nhiều–một.

Các trường dẫn xuất gồm GDP/người, log tự nhiên của GDP/người, phần trăm thay đổi GDP/người giữa các dòng kế tiếp của từng ISO3 và các cờ độ phủ. Phần trăm thay đổi dùng GDP theo USD hiện hành, nên không thể đọc trực tiếp là tăng trưởng kinh tế thực; nếu thiếu một dòng năm, hai dòng được so sánh có thể không cách nhau đúng một năm.

Lạm phát được chuẩn hóa bằng z-score trong từng năm; `inflation_extreme_flag` đánh dấu `|z| > 4` nhưng vẫn giữ quan sát. Cờ `high_inflation_ge_10pct` đánh dấu lạm phát từ 10% trở lên theo quy ước mô tả của dự án.

Kết quả được lưu thành `dim_country.csv` và `fact_country_yearly.csv`. Hai báo cáo `country_codes_excluded.csv` và `country_coverage_by_year.csv` lần lượt ghi các mã không ánh xạ được và độ phủ từng chỉ tiêu theo năm. Khi so sánh lạm phát giữa các nền kinh tế, cần xét khác biệt về rổ hàng hóa và phương pháp thống kê; các biểu đồ sau chỉ phục vụ mô tả.

In [3]:
inflation = pd.read_csv(RAW / "country" / "wb_cpi.csv").rename(
    columns={"Country Code": "iso3", "Year": "year", "CPI": "inflation_annual_pct"}
)[["iso3", "year", "inflation_annual_pct"]]
gdp = pd.read_csv(RAW / "country" / "wb_gdp.csv").rename(
    columns={"Country Code": "iso3", "Year": "year", "Value": "gdp_current_usd"}
)[["iso3", "year", "gdp_current_usd"]]
population = pd.read_csv(RAW / "country" / "wb_population.csv").rename(
    columns={"Country Code": "iso3", "Year": "year", "Value": "population"}
)[["iso3", "year", "population"]]

for name, frame in {"inflation": inflation, "gdp": gdp, "population": population}.items():
    frame["year"] = pd.to_numeric(frame["year"], errors="coerce").astype("Int64")
    assert_unique_key(frame, ["iso3", "year"], name)

panel_all = inflation.merge(
    gdp, on=["iso3", "year"], how="outer", validate="one_to_one", indicator="_merge_inflation_gdp"
)
panel_all = panel_all.merge(
    population, on=["iso3", "year"], how="outer", validate="one_to_one", indicator="_merge_population"
)
unmatched = panel_all.loc[~panel_all["iso3"].isin(dim_country["iso3"])].copy()
panel = panel_all.loc[panel_all["iso3"].isin(dim_country["iso3"])].copy()
panel = panel.loc[panel["year"].between(2013, 2024)].drop(
    columns=["_merge_inflation_gdp", "_merge_population"]
)

panel["inflation_annual_pct"] = pd.to_numeric(panel["inflation_annual_pct"], errors="coerce")
panel["gdp_current_usd"] = pd.to_numeric(panel["gdp_current_usd"], errors="coerce").where(lambda s: s > 0)
panel["population"] = pd.to_numeric(panel["population"], errors="coerce").where(lambda s: s > 0)
panel["gdp_per_capita_usd"] = panel["gdp_current_usd"] / panel["population"]
panel["log_gdp_per_capita"] = np.log(panel["gdp_per_capita_usd"].where(lambda s: s > 0))
panel = panel.sort_values(["iso3", "year"])
panel["gdp_per_capita_growth_pct"] = (
    panel.groupby("iso3")["gdp_per_capita_usd"].pct_change(fill_method=None) * 100
)
panel["inflation_year_z"] = panel.groupby("year")["inflation_annual_pct"].transform(
    lambda s: (s - s.mean()) / s.std(ddof=0)
)
panel["inflation_extreme_flag"] = panel["inflation_year_z"].abs().gt(4).where(
    panel["inflation_year_z"].notna()
).astype("boolean")
panel["high_inflation_ge_10pct"] = panel["inflation_annual_pct"].ge(10).where(
    panel["inflation_annual_pct"].notna()
).astype("boolean")
panel["has_inflation"] = panel["inflation_annual_pct"].notna().astype("Int64")
panel["has_gdp"] = panel["gdp_current_usd"].notna().astype("Int64")
panel["has_population"] = panel["population"].notna().astype("Int64")
panel = panel.merge(dim_country, on="iso3", how="left", validate="many_to_one")
panel = panel.sort_values(["iso3", "year"])
assert_unique_key(panel, ["iso3", "year"], "fact_country_yearly")
assert panel["continent_code"].notna().all()

In [4]:
write_csv_atomic(dim_country, TABLES / "dim_country.csv")
write_csv_atomic(panel, TABLES / "fact_country_yearly.csv")
write_csv_atomic(unmatched, QUALITY / "country_codes_excluded.csv")
coverage = panel.groupby("year").agg(
    economies=("iso3", "nunique"),
    inflation_available=("has_inflation", "sum"),
    gdp_available=("has_gdp", "sum"),
    population_available=("has_population", "sum"),
    inflation_extremes_retained=("inflation_extreme_flag", "sum"),
).reset_index()
write_csv_atomic(coverage, QUALITY / "country_coverage_by_year.csv")
print(f"Panel: {len(panel):,} rows, {panel.iso3.nunique()} ISO3; aggregate/unmapped codes excluded: {unmatched.iso3.nunique()}")
display(coverage.tail())

Panel: 2,580 rows, 215 ISO3; aggregate/unmapped codes excluded: 50


,year,economies,inflation_available,gdp_available,population_available,inflation_extremes_retained
7,2020,215,175,207,215,1
8,2021,215,175,206,215,2
9,2022,215,174,200,215,3
10,2023,215,170,184,215,1
11,2024,215,124,0,215,1
